# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagnik556/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Ranking.

The task is not to classify a product into a fixed category, and not to
cluster similar products together — it's to order a list of products/niches
from "most worth pursuing" to "least worth pursuing" for content or
dropshipping decisions. Ranking fits because the end action is a prioritized
list (top products to feature this week), not a single yes/no label.

In [5]:
import pandas as pd

path = "/content/flyrank-ml-track/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("First 10 columns:", df.columns.tolist()[:10])

Rows: 30000
Columns: 44
First 10 columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [6]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


Target: future content-performance decline.

For the modelling task, the target will be defined from a future performance window rather than from the same data used to build the features. A page will be considered a positive case when its future performance shows a meaningful decline according to a predefined rule. This makes the target an observed future outcome rather than a manually invented score. The current dataset's trend fields will be used carefully during exploration and will not be used as leakage-prone features when they contain information from the target window.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [7]:
from sklearn.metrics import average_precision_score

print("Success metrics selected:")
print("- Precision@K")
print("- PR-AUC")

Success metrics selected:
- Precision@K
- PR-AUC


Success metric: Precision@K, with PR-AUC as a secondary metric.

Precision@K measures how many of the pages in the model's highest-priority recommendations actually become future decline cases. This matches the real decision because an editor may only have time to review the top K pages. PR-AUC will provide an additional measure of ranking quality when the positive class is imbalanced. The model will be compared with a simple baseline on the same held-out split.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
sample = df.sample(5, random_state=42)

print("Sample rows:")
display(sample)

Sample rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
2308,content_9824710082d8,client_3fdba35f04,0.0,0.0,LOW,0.00,keyword article,informational,1397.0,9273.0,...,8000-15000,0.00,21.3,0.00,0.00,0.0,low,page_3_5,stable,8.0
22404,content_3efa3a7c46bb,client_f74efabef1,0.0,0.0,LOW,0.00,keyword article,informational,3188.0,22026.0,...,15000-25000,0.09,8.8,4.35,4.35,0.0,good,page_1,up,22.9
23397,content_575dc8a2ab0f,client_25fc0e7096,NaN,NaN,NaN,NaN,feedly article,NaN,3381.0,21992.0,...,15000-25000,0.00,0.0,0.00,20.83,0.0,low,top_3,new,NaN
25058,content_0dbd6911ba04,client_d029fa3a95,0.0,0.0,LOW,0.00,comparison article,informational,2892.0,19212.0,...,15000-25000,0.00,8.1,0.00,40.00,0.0,low,page_1,down,-57.4
2664,content_bbaf87019afb,client_19581e27de,30.0,1.0,HIGH,1.52,keyword article,commercial,NaN,NaN,...,NaN,0.23,30.9,2.33,2.08,0.0,good,page_3_5,down,-20.8


One row represents one content-page observation in the starter dataset. Each observation contains anonymized content and search-performance signals that describe the page's current state. The modelling task will use these observations to construct features from an appropriate historical window and evaluate the page against a future performance outcome.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [9]:
print("Candidate modelling inputs will be selected after the data and leakage audit.")
print("The baseline will be defined before comparing it with ML models.")

Candidate modelling inputs will be selected after the data and leakage audit.
The baseline will be defined before comparing it with ML models.


A fixed rule may miss useful combinations of signals because content performance can depend on several factors at the same time. For example, a page with high visibility but declining clicks may deserve attention for a different reason than a page with low visibility and a recent recovery pattern. A model can learn relationships across multiple observable signals instead of relying on one manually chosen threshold. The model will still be treated as decision support, and its performance will be compared against a simple baseline rather than assuming that ML is automatically better.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.